# Supplementary Figure 8 — rats: SWRs, replay and BARRs, non-overlapping windows

Extended Data Fig. 6 of the first submission with every window statistic on non-overlapping windows (`ed_nonoverlap.py`, `ed_nonoverlap_rats_tiles.py`); the window panels are the first-submission rat notebook (Previous_Rv1 `FigS5_S6_rats`) run on those tables. d2 = d2 (2024 Euclidean, AR p = 20); rat sessions keep their bin sizes (40–80 ms) and window lengths (30 / 60 s).

- **a** Schematic; d2 presleep vs post-task sleep (NREM, session means), 17 sessions from 2 rats; session-paired signed-rank.
- **b** Branching ratio around SWR starts (example animal, SWRs vs shuffle, ± SEM) and per-session ΔBR inset; inset: each session's mean ΔBR around SWRs vs shuffled times, session-paired signed-rank (`rat_br_change_per_swr.csv`).
- **c** d2 (z within file) by replay / non-replay SWR-rate sextile, 15 sessions from 4 rats; mean ± SE; legend slopes = linear trend over the 6 sextile means for each arm and their difference.
- **d** ISI entropy (z within file) by SWR-rate sextile, 25 sessions from 4 rats, mean ± SEM; per-session Spearman ρ, signed-rank across sessions.
- **e** Branching ratio around BARR starts and per-session ΔBR inset, tested as in b (`rat_br_change_per_event_nonoverlap.csv`).
- **f** d2 (z) and ISI entropy (z) by BARR-rate sextile, 12 sessions from 4 rats, mean ± SEM over sessions; linear trend over the 6 sextile means.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))   # notebooks/figures_revision
import final_figs as F                     # paths, save/copy helpers; adds the revision folder to sys.path

## Data and statistics (a, d per session; c, f from the run's log)

In [ ]:
import re
import numpy as np
import pandas as pd
from scipy import stats
import fig3_background_repro as M3          # session_rho (Fig 3e / 3h)

# Supplementary Fig 8 = the submitted Extended Data Fig. 6 (rats) on non-overlapping windows:
#   a  d2 presleep vs postsleep        Previous_Rv1 FigS5_S6_rats summary_df (session means), d2 only
#   b  BR around SWRs + dBR            cut from Extended Data Fig6.svg (event-level)
#   c  d2 by replay / non-replay SWR-rate sextile   FigS5_S6_rats (tiles rebuilt by ed_nonoverlap_rats_tiles.py)
#   d  ISI entropy by SWR-rate sextile FigS5_S6_rats (thinned, re-tiled, re-z'd on disjoint windows)
#   e  BR around BARRs + dBR           submitted, cut (as b)
#   f  d2 and ISI entropy by BARR-rate sextile   FigS5_S6_rats (tables rebuilt by ed_nonoverlap_rats_tiles.py)
# Statistics: a session-paired signed-rank (the d2 test added here); d per session (as Supplementary
# Fig 4e, f): Spearman rho(value, rate) over each session's windows, signed-rank of the rhos across sessions
# (fig3_background_repro.session_rho, as Fig 3e, h); c and f: the
# notebook's own linear trend over the 6 sextile means (n = 6), read from the run's log.
LOG = os.path.join(F.REPO, "data", "revision", "log")
T = {"a": os.path.join(LOG, "figs8a_rat_prepost_disjoint.csv"), "d": os.path.join(LOG, "figs8d_rat_isi_by_swr_rate_disjoint.csv")}
JOBS = {"a": ('draw_paired_lines(ax, summary_df, y_col="d2", order=order)', "summary_df"),
        "d": ("df_ripple_pooled_isi = read_csv('rat_isi_entropy_by_swr_rate.csv')", "df_ripple_pooled_isi")}
for k, (after, var) in JOBS.items():
    if not os.path.exists(T[k]):
        F.rv1_table("FigS5_S6_rats", after, var, T[k])

# ---- a: one d2 per session and epoch (NREM; presleep vs the first post-task sleep), session-paired signed-rank
A = pd.read_csv(T["a"])
PA = A.pivot(index="animal_day", columns="epoch2", values="d2")[["Presleep", "Postsleep"]].dropna()
dA = PA.Postsleep - PA.Presleep
ST = {"a": dict(n=len(PA), up=int((dA > 0).sum()), direction="↑", mean_diff=float(dA.mean()),
                p=float(stats.wilcoxon(PA.Postsleep, PA.Presleep).pvalue))}


def per_session(df, sess, value, rate):
    rho, p = M3.session_rho(df, sess, value, rate)
    up = np.median(rho) > 0
    return dict(n=len(rho), up=int((rho > 0).sum()) if up else int((rho < 0).sum()), direction="↑" if up else "↓",
                rho_median=float(np.median(rho)), p=float(p))


# ---- d: ISI entropy (z within file) vs SWR rate, per session, on the notebook's thinned windows
D = pd.read_csv(T["d"])
ST["d"] = per_session(D, "file_id", "isi_entropy_norm", "ripple_rate")

# ---- c, f: the notebook's statistics, as it prints them in the run whose figures are drawn (panels cell);
# f's per-session Spearman is reported in the legend (ED6f / ED6g).
for k, name, col in (("f_d2", "rat_d2_by_barrage_rate.csv", "d2_norm"),
                     ("f_isi", "rat_isi_entropy_by_barrage_rate.csv", "isi_entropy_norm")):
    W = pd.read_csv(os.path.join(LOG, "ed_nonoverlap_tables", name))
    ST[k + "_per_session"] = per_session(W, "file_id", col, "barrage_rate")
# ---- b, e insets: per session, as Fig 3f's inset -- each
# session's mean dBR around events vs around shuffled times, session-paired signed-rank.
# Inputs: the notebook's own CSVs.
import ed_nonoverlap as E
INSET = []
for k, name, ev, sh in (("b", "rat_br_change_per_swr.csv", "Ripple", "Ripple_Shuffle"),
                        ("e", "rat_br_change_per_event_nonoverlap.csv", "Barrage", "Barrage_Shuffle")):
    ev_ = pd.read_csv(E.RV1 / "csv" / "FigS5_S6_rats" / name, float_precision="round_trip")
    ev_["file"] = ev_["animal"].astype(str) + "_" + ev_["day"].astype(str)
    ev_ = ev_.dropna(subset=["value", "animal", "day", "condition"])
    P = ev_.groupby(["file", "condition"]).value.mean().unstack()[[ev, sh]].dropna()
    dd = P[ev] - P[sh]
    up = np.median(dd) > 0
    ST[k] = dict(n=len(P), up=int((dd > 0).sum()) if up else int((dd < 0).sum()), direction="↑" if up else "↓",
                 p=float(stats.wilcoxon(P[ev], P[sh]).pvalue))
    nev = ev_.groupby(["file", "condition"]).size().unstack()
    INSET.append(pd.DataFrame({"panel": k, "source_csv": name, "session": P.index, "dBR_event": P[ev].to_numpy(),
                               "dBR_shuffle": P[sh].to_numpy(), "diff": dd.to_numpy(),
                               "n_events": nev.loc[P.index, ev].to_numpy(), "n_shuffle": nev.loc[P.index, sh].to_numpy(),
                               "p_signrank_paired": ST[k]["p"]}))
# the per-session table behind the b, e inset stars
pd.concat(INSET).to_csv(F.assert_local_path(os.path.join(LOG, "figs8_br_inset_sessions.csv")), index=False)
print(f"a: post vs pre, two-sided paired Wilcoxon signed-rank p = {ST['a']['p']:.2g}; n = {ST['a']['n']} sessions")
for k, what in (("b", "SWRs vs shuffle"), ("e", "BARRs vs shuffle")):
    print(f"{k} inset: {what}, two-sided paired Wilcoxon signed-rank p = {ST[k]['p']:.2g}; n = {ST[k]['n']} sessions")
print(f"c: slope {ST['c']['slope_in']:.2f} (replay SWRs), {ST['c']['slope_out']:.2f} (non-replay SWRs); slope difference "
      f"{-ST['c']['diff']:.2f}, p = {ST['c']['p_diff']:.2g}")
print(f"d: per-session Spearman rho, two-sided Wilcoxon signed-rank p = {ST['d']['p']:.2g}; n = {ST['d']['n']} sessions")
for k in ("d2", "isi"):
    print(f"f {k}: six sextile means, Pearson r = {ST[f'f_{k}']['r']:.2f}, p = {ST[f'f_{k}']['p']:.2g}")

## Panels — b, e and the schematic in a from Extended Data Fig. 6; c, d, f from the notebook

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

ED6 = os.path.join(F.REPO, "reports", "new_planc", "nature", "Extended Data Fig6.svg")   # the submitted figure, mm units

# ---- a (top), b, e: cropped from the submitted figure (b, e: event-level BR around SWRs / BARRs, the example
# animal and the per-session dBR inset)
F.crop_svg(ED6, "FigS8_A_schematic_prepost", 5.5, 4.3, 31.5, 12.6, unit_mm=1.0)
F.crop_svg(ED6, "FigS8_B_br_around_swr", 39.6, 0.6, 46.0, 55.8, drop_ids=["text910277-5"], unit_mm=1.0)   # 1.2 mm headroom above the inset bracket
F.crop_svg(ED6, "FigS8_E_br_around_barr", 8.6, 59.6, 61.0, 45.6, drop_ids=["text910277-5-16-1-1-1"], unit_mm=1.0)


def edit_crop(fig_id, formal):
    """Replace tspan texts {id: text or (text, y)} in a cut panel; a y moves the tspan's baseline (its own units)."""
    import subprocess
    src = open(F.stem(fig_id) + ".svg", encoding="utf-8").read()
    for variant, edits in ((None, formal),):
        s = src
        for i, txt in edits.items():
            txt, y = txt if isinstance(txt, tuple) else (txt, None)
            m = re.search(r'(<tspan\b[^>]*\bid="%s"[^>]*>)[^<]*(</tspan>)' % re.escape(i), s)
            assert m, (fig_id, i)
            tag = m.group(1) if y is None else re.sub(r'\by="[^"]*"', f'y="{y}"', m.group(1))
            s = s[:m.start()] + tag + txt + m.group(2) + s[m.end():]
            if y is not None:                          # a sodipodi:role="line" tspan is placed from its <text>'s y
                t0 = s.rfind("<text", 0, m.start())
                t1 = s.index(">", t0)
                s = s[:t0] + re.sub(r'\by="[^"]*"', f'y="{y}"', s[t0:t1]) + s[t1:]
        out = F.stem(fig_id) if variant is None else os.path.join(os.path.dirname(F.stem(fig_id)), variant, fig_id)
        os.makedirs(os.path.dirname(out), exist_ok=True)
        open(F.assert_local_path(out + ".svg"), "w", encoding="utf-8").write(s)
        subprocess.run([F.INKSCAPE, out + ".svg", "--export-type=png", "--export-dpi=600", "--export-background=#ffffff",
                        f"--export-filename={out}.png"], capture_output=True)


# the insets' stars -> the per-session test (data cell); e's bottom y tick is set to 0.85, evenly spaced with
# the other three (0.88 / 0.87 / 0.86)
sb, se = ST["b"], ST["e"]
edit_crop("FigS8_B_br_around_swr", {"tspan76756-5": F.stars(sb["p"])})
edit_crop("FigS8_E_br_around_barr", {"tspan76756-5-1": F.stars(se["p"]), "tspan83062-6-0": "0.85"})

# ---- a: d2 presleep vs postsleep, session means (NREM), as Supplementary Fig 4a / Fig 2a
F.final_style()
S = PA.rename_axis("session").reset_index().melt(id_vars="session", var_name="epoch2", value_name="d2")
order, pal = ["Presleep", "Postsleep"], {"Presleep": "#f6d5ff", "Postsleep": "#dd55ff"}
fig, ax = plt.subplots(figsize=(31.5 * F.MM, 41 * F.MM), layout="constrained")
sns.boxplot(data=S, x="epoch2", y="d2", order=order, hue="epoch2", hue_order=order, palette=pal, legend=False,
            showfliers=False, width=0.5, linewidth=0.6, ax=ax, zorder=2)
sns.stripplot(data=S, x="epoch2", y="d2", order=order, color="k", jitter=False, alpha=0.75, size=2.2, ax=ax, zorder=3)
for _, r in PA.iterrows():
    ax.plot([0, 1], [r.Presleep, r.Postsleep], color="gray", alpha=0.5, lw=0.5, zorder=2)
s = ST["a"]
top = S.d2.max()
span = top - S.d2.min()
F.bracket(ax, 0, 1, top + 0.08 * span, f"{s['up']}/{s['n']} ↑ {F.stars(s['p'])}", h=0.03 * span, fs=5.5)
ax.set_ylim(S.d2.min() - 0.05 * span, top + 0.3 * span)
ax.set_xlabel(""); ax.set_ylabel("d2")
F.settle(fig)
F.drop_counts(fig)
F.show(F.save(fig, "FigS8_A_prepost_d2"))

# ---- c, d, f: the rat notebook's own figures, run on disjoint windows (as ed_nonoverlap.py) and restyled to
# print size
KEYS = {"c": "figS6_C_rat_replay_non_replay_mean_sem_across_files", "d": "figS6_G_Rat_isi_entropy_by_d2_tile_pooled",
        "f_d2": "figS6_F_barrage_rate_by_d2_tile_pooled", "f_isi": "figS6_G_barrage_rate_by_isi_tile_pooled"}
CAP = F.run_rv1_capture("FigS5_S6_rats", list(KEYS.values()))
LOGTXT = open(F.LAST_RV1_LOG, encoding="utf-8").read()
_c = {k: eval(re.search(rf"^{k}\s*: (\{{.*\}})$", LOGTXT, flags=re.M).group(1)) for k in ("IN", "OUT")}   # noqa: S307
_d = re.search(r"Slope difference \(OUT-IN\): (\S+) .*?p=([\d.e-]+)", LOGTXT)
ST["c"] = dict(slope_in=_c["IN"]["b"], p_in=_c["IN"]["p_b"], slope_out=_c["OUT"]["b"], p_out=_c["OUT"]["p_b"],
               diff=float(_d.group(1)), p_diff=float(_d.group(2)))
for k, tab in (("f_d2", "rat_d2_by_barrage_rate.csv"), ("f_isi", "rat_isi_entropy_by_barrage_rate.csv")):
    blk = LOGTXT[LOGTXT.index(f"[override] {tab}"):]
    ST[k] = dict(r=float(re.search(r"r\s*=\s*([-\d.]+)", blk).group(1)), p=float(re.search(r"p = ([\d.e-]+)", blk).group(1)))
print("c", ST["c"]); print("f", ST["f_d2"], ST["f_isi"])


def drawn(ax):
    return [l_ for l_ in ax.get_lines() if len(l_.get_xdata()) >= 6]


# d: per-file tile means, then mean over files (the notebook's crossfileSEM); f: mean over all windows per tile
_pf = D.groupby(["file_id", "tile"]).isi_entropy_norm.mean().unstack().mean()
assert np.allclose(drawn(CAP[KEYS["d"]].axes[0])[0].get_ydata(), _pf.to_numpy()), "d"
for k, tab, col in (("f_d2", "rat_d2_by_barrage_rate.csv", "d2_norm"), ("f_isi", "rat_isi_entropy_by_barrage_rate.csv", "isi_entropy_norm")):
    _m = pd.read_csv(os.path.join(LOG, "ed_nonoverlap_tables", tab)).groupby("tile")[col].mean()
    assert np.allclose(drawn(CAP[KEYS[k]].axes[0])[0].get_ydata(), _m.to_numpy()), k


def finish(fig, fid, w, h, xl, yl, text=None, color="k", where=(0.5, 0.97), ha="center"):
    ax = fig.axes[0]
    F.restyle(fig, w, h)
    ax.set_title("")
    if ax.get_legend() and fid != "FigS8_C_d2_by_replay_swr_rate":
        ax.get_legend().remove()                       # "mean ± SEM (across files)": the legend's text
    ax.set_xticks(range(1, 7)); ax.set_xticklabels([str(i) for i in range(1, 7)])
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    if text:
        ax.text(*where, text, transform=ax.transAxes, ha=ha, va="top", fontsize=6, color=color)
    F.sparse_yticks(ax)
    F.settle(fig)
    F.drop_counts(fig)
    F.show(F.save(fig, fid))


# c: replay (IN) vs non-replay (OUT) SWR rate; the legend carries each arm's slope over the 6 sextile means
fig = CAP[KEYS["c"]]
ax = fig.axes[0]
for h_ in ax.get_legend_handles_labels()[0]:
    lab = h_.get_label()
    h_.set_label(f"Replay SWRs, slope = {ST['c']['slope_in']:.2f}" if lab == "IN" else
                 f"Non-replay SWRs, slope = {ST['c']['slope_out']:.2f}" if lab == "OUT" else lab)
ax.set_xticks(range(1, 7))
ax.legend(frameon=False, loc="upper left", handlelength=1.6)
finish(fig, "FigS8_C_d2_by_replay_swr_rate", 50, 50, "SWR-rate sextile", "d2 (z)")

s = ST["d"]
finish(CAP[KEYS["d"]], "FigS8_D_isi_entropy_by_swr_rate", 42, 50, "SWR-rate sextile", "ISI entropy (z)",
       f"{s['up']}/{s['n']} {s['direction']} {F.stars(s['p'])}", color="tab:red", where=(0.95, 0.97), ha="right")
finish(CAP[KEYS["f_d2"]], "FigS8_F_d2_by_barr_rate", 50, 44, "BARR-rate sextile", "d2 (z)",
       F.stars(ST["f_d2"]["p"]), color="tab:blue", where=(0.6, 0.97))
finish(CAP[KEYS["f_isi"]], "FigS8_F_isi_entropy_by_barr_rate", 50, 44, "BARR-rate sextile", "ISI entropy (z)",
       F.stars(ST["f_isi"]["p"]), color="tab:blue", where=(0.4, 0.97))